# Module 2 lab: a predictable local HTTP contract

No server or network is used; dictionaries model HTTP messages.

## Objectives and predictions

Build a router, bounded collection, stable errors, and idempotent create. Predict 1: status for missing task? Predict 2: IDs after a limit-2 sorted page? Predict 3: how many effects after replaying a key?

In [ ]:
tasks=[{"id":2,"owner":"ana","title":"Read"},{"id":1,"owner":"ben","title":"Test"}]; created={}
def error(status,code,message):
 return {"status":status,"headers":{"Content-Type":"application/json"},"body":{"error":{"code":code,"message":message}}}
def list_tasks(q):
 try: limit=int(q.get("limit","20")); offset=int(q.get("offset","0"))
 except ValueError: return error(400,"invalid_paging","paging must be integers")
 if not 1<=limit<=50 or offset<0: return error(400,"invalid_paging","limit 1..50 and offset >= 0")
 chosen=sorted([t for t in tasks if not q.get("owner") or t["owner"]==q["owner"]],key=lambda t:t["id"])
 return {"status":200,"headers":{"Content-Type":"application/json"},"body":{"items":chosen[offset:offset+limit],"limit":limit,"offset":offset}}
def create_task(body,headers):
 key=headers.get("Idempotency-Key"); fp=(body.get("owner"),body.get("title"))
 if not body.get("title") or not body.get("owner"): return error(400,"missing_field","title and owner are required")
 if key in created:
  if created[key][0]!=fp: return error(409,"idempotency_conflict","key has different content")
  return created[key][1]
 task={"id":max([t["id"] for t in tasks],default=0)+1,"owner":body["owner"],"title":body["title"]}; tasks.append(task)
 response={"status":201,"headers":{"Content-Type":"application/json"},"body":task}
 if key: created[key]=(fp,response)
 return response
baseline={"status":200,"body":tasks}
page=list_tasks({"limit":"2"}); assert [x["id"] for x in page["body"]["items"]]==[1,2]
a=create_task({"owner":"ana","title":"Ship"},{"Idempotency-Key":"k"}); b=create_task({"owner":"ana","title":"Ship"},{"Idempotency-Key":"k"})
assert a==b and len([t for t in tasks if t["title"]=="Ship"])==1

## Prediction answers

1. A missing task is 404 because the request is shaped correctly but the resource is absent. 2. The bounded page is ordered by IDs 1 and 2. 3. Replaying the same key creates one effect and returns the stored result.

## Router and status classes

A router extracts path meaning; a service chooses the result. Missing resources are absence, malformed identifiers are caller errors, and a route miss has its own stable code.

In [ ]:
def get_task(i):
 for t in tasks:
  if t["id"]==i: return {"status":200,"headers":{"Content-Type":"application/json"},"body":t}
 return error(404,"not_found","task was not found")
def route(req):
 m,p=req["method"],req["path"]
 if m=="GET" and p=="/tasks": return list_tasks(req.get("query",{}))
 if m=="POST" and p=="/tasks": return create_task(req.get("body") or {},req.get("headers",{}))
 if m=="GET" and p.startswith("/tasks/"):
  try: return get_task(int(p.split("/")[-1]))
  except ValueError: return error(400,"invalid_id","task ID must be an integer")
 return error(404,"route_not_found","route was not found")
assert route({"method":"GET","path":"/tasks/999"})["status"]==404
assert route({"method":"GET","path":"/tasks/x"})["status"]==400
assert route({"method":"POST","path":"/tasks","body":{},"headers":{}})["status"]==400

## AI-style critique and guided TODO

The generated pattern below is plausible but unbounded and assumes every POST can be retried. Predict the review flags, then run the safe reference.

In [ ]:
ai_style="return {'status':200,'body':tasks}; retry POST whenever status != 200"
assert "body':tasks" in ai_style and "retry POST" in ai_style
print("flags: unbounded list, incorrect side-effect assumption, no replay key")
def bounded_owner_page(records,owner): return sorted((r for r in records if r["owner"]==owner),key=lambda r:r["id"])[:2]
assert [r["id"] for r in bounded_owner_page(tasks,"ana")]==sorted(r["id"] for r in tasks if r["owner"]=="ana")[:2]

## Independent challenge

Add an allow-listed title sort with id as tie-breaker. Exit answers: 409 is a state/key conflict; a server clamp proves a bound; this lab does not prove framework, proxy, network, or distributed behavior.

Evidence: keep predictions, baseline, captures, contract table, positive/negative/failure checks, replay count, and AI review.

## Baseline reproduction: slow path
The original baseline returns every record and does not say which content type it represents. Reproduce that observable behavior before editing; this is a characterization step.

In [ ]:
baseline_count=len(baseline["body"])
assert baseline_count >= 3
print("baseline returned",baseline_count,"records with no server-enforced bound")

## Pre-edit hypothesis
Write this before changing the design: “If the service clamps limit to 50, sorts by id, and applies owner filtering before slicing, then a caller cannot request an unbounded page and repeated reads have deterministic order.”

In [ ]:
pre_edit_hypothesis="clamp, filter, and sort will make collection behavior bounded and deterministic"
assert "bounded" in pre_edit_hypothesis or "clamp" in pre_edit_hypothesis

## Incremental guided implementation
First parse one query value at the boundary. Then use the parsed value in a collection service. Keeping these steps separate makes malformed input a 400 instead of an accidental exception.

In [ ]:
def parse_limit(raw):
 try: value=int(raw)
 except (TypeError,ValueError): return None
 return value if 1<=value<=50 else None
assert parse_limit("10")==10 and parse_limit("0") is None and parse_limit("x") is None

In [ ]:
def reference_list(q):
 limit=parse_limit(q.get("limit","20"))
 if limit is None: return error(400,"invalid_limit","limit must be 1..50")
 items=sorted((t for t in tasks if not q.get("owner") or t["owner"]==q["owner"]),key=lambda t:t["id"])
 return {"status":200,"body":{"items":items[:limit],"limit":limit}}
assert reference_list({"owner":"ana","limit":"1"})["body"]["items"][0]["owner"]=="ana"

## Positive, negative, and failure checks
A positive check proves a valid read. Negative checks prove caller errors and absence. The local failure case below models a service exception without contacting anything.

In [ ]:
assert reference_list({"limit":"2"})["status"]==200
assert reference_list({"limit":"999"})["status"]==400
assert route({"method":"GET","path":"/tasks/999"})["status"]==404
try: raise RuntimeError("synthetic store failure")
except RuntimeError: failure=error(500,"internal_error","request failed")
assert failure["status"]==500 and "RuntimeError" not in str(failure)

## AI-style/broken-code critique
The generated snippet confuses transport success with business success and has no replay boundary. Identify those flaws before looking at the safe implementation.

In [ ]:
broken_api="return 200, all_tasks(); except Exception: return 200, {'ok': True}"
assert "all_tasks" in broken_api and "'ok': True" in broken_api
print("Reject: unbounded output and false success after failure.")

## Guided TODO: attempt
Attempt the task yourself: accept only sort=id or sort=title, then use id as a tie-breaker. This attempt is executable and intentionally small; compare it with the reference below.

In [ ]:
allowed_sort={"id","title"}
def todo_sort(records, requested):
 field=requested if requested in allowed_sort else "id"
 return sorted(records,key=lambda r:(r[field],r["id"]))
assert todo_sort([{"id":2,"title":"A"},{"id":1,"title":"A"}],"title")[0]["id"]==1

## Reference solution
The reference makes the allow-list and tie-breaker explicit; it does not evaluate a caller-provided expression.

In [ ]:
def reference_sort(records, requested):
 field=requested if requested in {"id","title"} else "id"
 return sorted(records,key=lambda r:(r[field],r["id"]))
assert reference_sort(tasks,"not-a-field")==sorted(tasks,key=lambda r:(r["id"],r["id"]))

## Independent challenge
Add a compatibility assertion for an original client that reads body.items and ignores the new limit metadata. Explain why removing items would be breaking even if status stays 200.

In [ ]:
old_client=lambda response:[item["id"] for item in response["body"]["items"]]
assert old_client(reference_list({"limit":"2"}))==[1,2]

## Exit questions and Answers
1. Why must the server enforce the limit? Because callers can omit or inflate it.
2. Why separate router and service? Each can be tested without the other.
3. What does an idempotency key protect here? A repeated local create effect, not distributed exactly-once.
4. Which change can break a strict client? Removing/renaming a field or adding rejected fields.

## Evidence handoff
Hand off your predictions and explanations, baseline capture, written hypothesis, incremental outputs, positive/negative/failure results, AI critique, TODO/reference comparison, challenge compatibility assertion, and a clean restart run.